## Example introduction into a client with o6-python

When using a notebook, you have to await the server responses
The first cell is executing the GUI, make sure the basic_sim_server python script is in the correct path.


In [ ]:
import subprocess
import sys
import socket
import time
from pathlib import Path

# Start the simulation server in a separate process so the notebook kernel stays responsive.
server_script = (Path.cwd() / "../sim_examples/server/basic_sim_server.py").resolve()
sim_proc = subprocess.Popen(
    [sys.executable, str(server_script)],
    cwd=str(server_script.parent),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
    text=True,
 )

# Wait until OPC UA endpoint is reachable before running client cells.
for _ in range(30):
    try:
        with socket.create_connection(('127.0.0.1', 4840), timeout=0.5):
            break
    except OSError:
        time.sleep(0.2)
else:
    raise RuntimeError('Server did not start on opc.tcp://localhost:4840 in time.')

print(f'Server started in background (PID {sim_proc.pid})')

Server started in background (PID 102600)


In [2]:
from o6 import Client, StatusCodeError, types
import time
client = Client("opc.tcp://localhost:4840")
await client.connect()

value: 0.95
value: -1.0
value: 0.55
value: -1.0
value: 0.4
value: -1.0
value: 0.95
value: -1.0
value: 0.85
value: -1.0
value: -0.9
value: -0.65
value: -0.4
value: -0.35
value: -0.25
value: -0.15
value: -0.05
value: 0.0
value: -0.05
value: -0.2
value: -0.25
value: -0.2
value: -0.15
value: -0.1
value: 0.0


client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subscription
client / Received BadNoSubscription, delete internal information about subsc

Get a list of all the objects on the server with browse on the root object with the nodeid ```"ns=0;i=85"```

```browse``` is a list of the reference objects  
```read``` is the value of the object

In [3]:
for item in await client.browse("ns=0;i=85"):
    print(item)

{reference_type_id=i=0, is_forward=False, nodeid=i=2253, browse_name=, display_name=, node_class=0, type_definition=i=0}
{reference_type_id=i=0, is_forward=False, nodeid=ns=1;s=PumpLIOpen, browse_name=, display_name=, node_class=0, type_definition=i=0}
{reference_type_id=i=0, is_forward=False, nodeid=ns=1;s=PumpLOOpen, browse_name=, display_name=, node_class=0, type_definition=i=0}
{reference_type_id=i=0, is_forward=False, nodeid=ns=1;s=PumpRIOpen, browse_name=, display_name=, node_class=0, type_definition=i=0}
{reference_type_id=i=0, is_forward=False, nodeid=ns=1;s=PumpROOpen, browse_name=, display_name=, node_class=0, type_definition=i=0}
{reference_type_id=i=0, is_forward=False, nodeid=ns=1;s=PumpCOpen, browse_name=, display_name=, node_class=0, type_definition=i=0}


In [4]:
value = await client.read("ns=1;s=PumpLIOpen")
print(value)

0.0


When writing a variable on the server, you need to use the o6-python types, ```Double, ... ```

When the Left Input is opened and closed, the simulation will respond.

In [5]:
await client.write("ns=1;s=PumpCOpen", types.Double(-0.0))


_o6.types.StatusCode(code=0x00, symbol=Good)

In [6]:
time.sleep(5)

In [7]:
await client.write("ns=1;s=PumpLIOpen", types.Double(0.0))

_o6.types.StatusCode(code=0x00, symbol=Good)

Stop the simulation cleanly when done:

In [8]:
# Stop background simulation server process when done.
def stop():
    if 'sim_proc' in globals() and sim_proc.poll() is None:
        sim_proc.terminate()
        try:
            sim_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            sim_proc.kill()
            sim_proc.wait(timeout=2)
        print('Simulation server stopped.')
    else:
        print('No running simulation server process found.')

In [12]:
await client.disconnect()
stop()

Simulation server stopped.
